# 3D Anomaly Detection Benchmark — Anomaly-ShapeNet + Real3D-AD

**Setup:**
1. Enable the **T4 GPU** accelerator (Settings → Accelerator)
2. **Add your datasets as Inputs** (Add Input → Your Work → your uploads). The
   notebook auto-detects them in `/kaggle/input` — upload either the extracted
   folders or the original archives (`Anomaly-ShapeNet-v2.rar`,
   `real3d-ad-pcd.zip`); both work. If an input is missing, the notebook falls
   back to downloading it automatically.
3. Set `REPO_URL` below to your GitHub fork of this project
4. *Run All*

Expected wall-clock: Anomaly-ShapeNet ~30–60 min · Real3D-AD ~2–4 h.
For full-benchmark runs you can use *Save Version → Save & Run All* so
everything persists even if you disconnect.

In [ ]:
%pip install -q open3d gdown
import torch
print('GPU available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# ---- get the ad3d codebase -------------------------------------------------
import os, glob

REPO_URL = "https://github.com/YOUR_USERNAME/ad3d.git"   # <-- EDIT: your fork

if not os.path.exists('ad3d'):
    if 'YOUR_USERNAME' not in REPO_URL:
        os.system(f'git clone --depth 1 {REPO_URL}')
    if not os.path.exists('ad3d'):
        # fallback: the folder was uploaded as a Kaggle Dataset (Add Input)
        hits = glob.glob('/kaggle/input/**/run.py', recursive=True)
        assert hits, 'Set REPO_URL to your GitHub fork, or upload the ad3d folder as a Kaggle input.'
        os.symlink(os.path.dirname(hits[0]), 'ad3d')
print('code at:', os.path.abspath('ad3d'))

In [ ]:
# ---- Anomaly-ShapeNet: auto-detect in /kaggle/input (fallback: download) ----
import os, glob, shutil, subprocess

DATA = '/kaggle/temp/ad3d-data' if os.path.exists('/kaggle/temp') else 'data'
os.makedirs(DATA, exist_ok=True)
SHAPENET_ROOT = f'{DATA}/anomaly-shapenet/pcd'      # used when we extract ourselves

if not os.path.exists(f'{SHAPENET_ROOT}/ashtray0/train'):
    hit = glob.glob('/kaggle/input/**/ashtray0/train', recursive=True)
    if hit:
        # dataset uploaded already-extracted -> use the read-only input in place
        SHAPENET_ROOT = os.path.dirname(os.path.dirname(hit[0]))
        print('using Anomaly-ShapeNet from Kaggle input:', SHAPENET_ROOT)
    else:
        # archive uploaded as input? else download from HuggingFace
        arcs = [p for p in (glob.glob('/kaggle/input/**/*.rar', recursive=True) +
                            glob.glob('/kaggle/input/**/*.zip', recursive=True))
                if any(k in os.path.basename(p).lower() for k in ('anomaly', 'shapenet'))]
        if not arcs:
            try:
                from kaggle_secrets import UserSecretsClient
                os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
                print('HF_TOKEN loaded from Kaggle secrets')
            except Exception:
                pass
            from huggingface_hub import hf_hub_download
            arcs = [hf_hub_download(repo_id='Chopper233/Anomaly-ShapeNet',
                                    filename='Anomaly-ShapeNet-v2.rar',
                                    repo_type='dataset', local_dir=f'{DATA}/hf_asn')]
        arc = arcs[0]
        print(f'archive: {arc} ({os.path.getsize(arc)/1e9:.2f} GB)')

        ext = f'{DATA}/asn_extracted'
        if arc.endswith('.zip'):
            import zipfile
            with zipfile.ZipFile(arc) as zf:
                zf.extractall(ext)
        else:   # .rar  (unar handles RAR5 + the Windows backslash paths)
            if shutil.which('unar') is None:
                subprocess.run(['apt-get', 'install', '-y', '-qq', 'unar'], check=True)
            subprocess.run(['unar', '-force-overwrite', '-quiet', '-o', ext, arc], check=True)

        hit = glob.glob(f'{ext}/**/ashtray0/train', recursive=True)
        assert hit, 'ashtray0/train not found after extraction - inspect ' + ext
        pcd_root = os.path.dirname(os.path.dirname(hit[0]))
        os.makedirs(os.path.dirname(SHAPENET_ROOT), exist_ok=True)
        if pcd_root != SHAPENET_ROOT:
            shutil.move(pcd_root, SHAPENET_ROOT)
        shutil.rmtree(ext, ignore_errors=True)
        shutil.rmtree(f'{DATA}/hf_asn', ignore_errors=True)

n_train = len(glob.glob(f'{SHAPENET_ROOT}/*/train/*.pcd'))
n_cat   = len(glob.glob(f'{SHAPENET_ROOT}/*/train'))
print(f'Anomaly-ShapeNet ready at {SHAPENET_ROOT} ({n_cat} categories, {n_train} training clouds)')

In [ ]:
# ---- Real3D-AD: auto-detect in /kaggle/input (fallback: download) -----------
import os, glob, shutil, subprocess, zipfile

DATA = '/kaggle/temp/ad3d-data' if os.path.exists('/kaggle/temp') else 'data'
os.makedirs(DATA, exist_ok=True)
REAL_ROOT = f'{DATA}/real3d-ad'

if not os.path.exists(f'{REAL_ROOT}/airplane/train'):
    hits = glob.glob('/kaggle/input/**/airplane/train', recursive=True)
    if hits:
        # dataset uploaded already-extracted -> use the read-only input in place
        REAL_ROOT = os.path.dirname(os.path.dirname(hits[0]))
        print('using Real3D-AD from Kaggle input:', REAL_ROOT)
    else:
        # zip uploaded as input? else download from Google Drive
        zips = [p for p in glob.glob('/kaggle/input/**/*.zip', recursive=True)
                if 'real3d' in os.path.basename(p).lower()]
        if not zips:
            rc = subprocess.run(
                ['gdown', '--fuzzy',
                 'https://drive.google.com/file/d/1oM4qjhlIMsQc_wiFIFIVBvuuR8nyk2k0/view?usp=sharing',
                 '-O', f'{DATA}/real3d-ad-pcd.zip']).returncode
            assert rc == 0 and os.path.exists(f'{DATA}/real3d-ad-pcd.zip'), (
                'gdown failed (Drive quota?). Upload real3d-ad-pcd.zip as a Kaggle input instead.')
            zips = [f'{DATA}/real3d-ad-pcd.zip']
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(f'{DATA}/real3d_raw')
        hits = glob.glob(f'{DATA}/real3d_raw/**/airplane/train', recursive=True)
        assert hits, 'airplane/train not found after extraction'
        cat_root = os.path.dirname(os.path.dirname(hits[0]))
        os.makedirs(os.path.dirname(REAL_ROOT), exist_ok=True)
        if cat_root != REAL_ROOT:
            shutil.move(cat_root, REAL_ROOT)
        shutil.rmtree(f'{DATA}/real3d_raw', ignore_errors=True)

n_train = len(glob.glob(f'{REAL_ROOT}/*/train/*.pcd'))
print(f'Real3D-AD ready at {REAL_ROOT} ({n_train} training clouds)')

In [ ]:
# ---- smoke test: 2 categories, small settings (a few minutes) ---------------
SMOKE = ('--num-group 512 --group-size 64 --max-nn 60 --n-scales 2 '
         '--points-budget 30000')
!python ad3d/run.py --dataset shapenet --data-root "{SHAPENET_ROOT}" --classes ashtray0,bowl0 {SMOKE} --tag smoke

In [ ]:
# ---- FULL Anomaly-ShapeNet benchmark (40 categories) -----------------------
!python ad3d/run.py --dataset shapenet --data-root "{SHAPENET_ROOT}" --tag simple3dlite

In [ ]:
# ---- FULL Real3D-AD benchmark (12 categories) -------------------------------
!python ad3d/run.py --dataset real3d --data-root "{REAL_ROOT}" --tag simple3dlite

In [ ]:
# ---- summarize --------------------------------------------------------------
import pandas as pd, glob
frames = []
for f in sorted(glob.glob('results/simple3dlite_*.csv')):
    df = pd.read_csv(f, index_col=0)
    df.index.name = 'category'
    df['source'] = f
    frames.append(df)
if frames:
    full = pd.concat(frames)
    means = full.groupby('source')[['o_auroc','p_auroc','p_aupr','p_auroc_pooled','p_aupr_pooled']].mean()
    display(means)
    full.drop(columns='source').to_csv('ad3d_results_summary.csv')
    print('saved ad3d_results_summary.csv')
else:
    print('no full-run results yet')

## Next steps

Open **`ad3d/NOVELTY_ROADMAP.md`** — it maps the open failure modes in these
benchmarks (point-level AUPR, the gemstone/starfish/duck categories, the
360°-train vs single-view-test gap) to the exact functions to modify:

| To change... | Edit... |
|---|---|
| features / descriptors | `src/ad3d/features.py` → `compute_fpfh_ms()` |
| grouping / aggregation | `src/ad3d/features.py` → `lfsa()` |
| memory bank / distance | `src/ad3d/memory.py` |
| scoring rules | `src/ad3d/scoring.py` |
| whole pipeline | `src/ad3d/method.py` → subclass `Simple3DLite` |

Every run writes its full config + per-category results into `results/*.json`,
so tagged comparisons are always reproducible.